In [ ]:
# Install the PyPharao code and py3Dmol
!pip install rdkit
!pip install git+https://github.com/silicos-it/PyPharao.git
!pip install mols2grid py3Dmol

In [ ]:
# Import all required modules
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import IPythonConsole


from IPython.display import display
from IPython.display import SVG

import py3Dmol
from ipywidgets import widgets

import mols2grid
from pypharao import *

from pathlib import Path
import requests

In [ ]:
# Get the PDB's: the pharmacophore PDB
URL = "https://raw.githubusercontent.com/silicos-it/PyPharao/master/examples/datasets/4HFZ_pharmacophore.pdb"
PROTEIN_PDB = Path("./4HFZ_pharmacophore.pdb")
print(f"Downloading {URL} to {PROTEIN_PDB}...")
response = requests.get(URL)
response.raise_for_status() # Raise an exception for HTTP errors
with open(PROTEIN_PDB, "wb") as f: f.write(response.content)
print(f"Successfully downloaded {PROTEIN_PDB.name} to {PROTEIN_PDB}")

# Get the PDB's: the exclusion PDB
URL = "https://raw.githubusercontent.com/silicos-it/PyPharao/master/examples/datasets/4HFZ_exclusion.pdb"
EXCL_PDB = Path("./4HFZ_exclusion.pdb")
print(f"Downloading {URL} to {EXCL_PDB}...")
response = requests.get(URL)
response.raise_for_status() # Raise an exception for HTTP errors
with open(EXCL_PDB, "wb") as f: f.write(response.content)
print(f"Successfully downloaded {EXCL_PDB.name} to {EXCL_PDB}")

In [ ]:
# Query from protein pharmacophore PDB + exclusion-sphere PDB
query = query_pharmacophore_from_protein(
    PROTEIN_PDB,
    EXCL_PDB,
    min_distance_between_excl_points=1.5,
    name="4HFZ-binding-site",
)

print(f"\nQuery {query.get_name()!r} ({len(query)} features):")
for p in query:
    print(f"  {p.type.value:<10} center={p.center}")

In [ ]:
mol_from_ph = query.to_mol()
view = py3Dmol.view(
    data=Chem.MolToMolBlock(mol_from_ph),  # Convert the RDKit molecule for py3Dmol
    style={"stick": {}, "sphere": {"scale": 0.3}}
)
view.zoomTo()

In [ ]:
# Write pharmacophore to pdb file
PHARMACOPHORE1_AS_PDB = Path("./ph1_from_4HFZ.pdb")
query.write_pdb(PHARMACOPHORE1_AS_PDB)

In [ ]:
# First merge all three LIPO points in a central one (new coordinate = -6.989, -32.071, 31.677)
# We will modify the first LIPO (i = 4), and remove i = 5 and 6

x = y = z = 0.0
for i, p in enumerate(query):
    if i in [4,5,6]:
        x += p.center[0]
        y += p.center[1]
        z += p.center[2]
x /= 3
y /= 3
z /= 3
query.update_point(4, center=(x,y,z))
query.remove_point(6)
query.remove_point(5)

In [ ]:
# We can also remove AROM 2
query.remove_point(1)

In [ ]:
# Now remove all EXCL points that are located than 8 A from any pharmacophore point
n_removed = query.purge_exclusion_spheres(distance=8)
print("Purged pharmacophore. Removed %d EXCL points" % (n_removed))

In [ ]:
# Write as pdb and print final set
PHARMACOPHORE2_AS_PDB = Path("./ph2_from_4HFZ.pdb")
query.write_pdb(PHARMACOPHORE2_AS_PDB)

In [ ]:
for i, p in enumerate(query):
    if i >= 8:
        print(f"  … ({len(query) - 12} more)")
        break
    label = p.type.value
    x, y, z = p.center
    coord = f"({x:.3f}, {y:.3f}, {z:.3f})"
    extra = f" σ={p.sigma:.2f}" if p.type == PointType.EXCL else ""
    print(f"  {label:<14} center={coord}{extra}")

In [ ]:
mol_from_ph = query.to_mol()
view = py3Dmol.view(
    data=Chem.MolToMolBlock(mol_from_ph),  # Convert the RDKit molecule for py3Dmol
    style={"stick": {}, "sphere": {"scale": 0.3}}
)
view.zoomTo()